In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
def print_title(text):
    print('='*50)
    print(text)
    print('='*50)

In [3]:
print_title('load the dataset')

df=sns.load_dataset('titanic')

load the dataset


In [7]:
print_title('MISSING VALUES IMPUTATION')

df_clean = df.copy()

df_clean['age_missing'] = df_clean['age'].isna()

df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())
df_clean['fare'] = df_clean['fare'].fillna(df_clean['fare'].median())

for column in ['embarked', 'embark_town', 'deck']:
    if df_clean[column].isna().any():
        df_clean[column] = df_clean[column].fillna(
            df_clean[column].mode()[0]
        )

df_clean.isna().sum()

MISSING VALUES IMPUTATION


survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
deck           0
embark_town    0
alive          0
alone          0
age_missing    0
dtype: int64

In [17]:
print_title('remove duplicates')
df_clean.drop_duplicates(inplace=True)
print('done')

remove duplicates
done


In [20]:
print_title('DATA TYPES')

for column in ['age', 'fare', 'sibsp', 'parch', 'pclass', 'survived']:
    df_clean[column] = pd.to_numeric( df_clean[column], errors='coerce' )

df_clean['survived'] = df_clean['survived'].astype('Int64')
df_clean['pclass'] = df_clean['pclass'].astype('category')

df_clean.dtypes

DATA TYPES


survived          Int64
pclass         category
sex              object
age             float64
sibsp             int64
parch             int64
fare            float64
embarked         object
class          category
who              object
adult_male         bool
deck           category
embark_town      object
alive            object
alone              bool
age_missing        bool
dtype: object

In [21]:
print_title('TEXT CLEANING')

for column in df_clean.select_dtypes(include='object').columns:
    df_clean[column] = ( df_clean[column] 
                        .str.strip() 
                        .str.lower() )

df_clean['sex'] = df_clean['sex'].replace({ 'm': 'male', 'f': 'female' })

df_clean['sex'].value_counts(dropna=False)

TEXT CLEANING


sex
male      490
female    293
Name: count, dtype: int64

In [22]:
print_title('OUTLIERS')

for column in ['age', 'fare']:
    Q1 = df_clean[column].quantile(0.25)
    Q3 = df_clean[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    iqr_outliers = ( (df_clean[column] < lower_bound) | 
                     (df_clean[column] > upper_bound)
    )

    z_scores = ( (df_clean[column] - df_clean[column].mean()) / 
                  df_clean[column].std()
    )

    z_outliers = z_scores.abs() > 3

    print(f'\n{column}')
    print(f'IQR outliers: {iqr_outliers.sum()}')
    print(f'Z-score outliers: {z_outliers.sum()}')

OUTLIERS

age
IQR outliers: 38
Z-score outliers: 4

fare
IQR outliers: 102
Z-score outliers: 20


In [25]:
print_title('REMOVE IMPOSSIBLE VALUES')

for column in ['age', 'fare', 'sibsp', 'parch']:
    df_clean.loc[df_clean[column] < 0, column] = np.nan
print('Done')

REMOVE IMPOSSIBLE VALUES
Done


In [26]:
df.columns

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone'],
      dtype='object')

In [28]:
print_title('CLEAN FUNCTION')

def clean(df):
    df = df.copy()

    # Rename columns to snake_case
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_')
    )

    # Clean text columns
    for column in df.select_dtypes(include='object').columns:
        df[column] = df[column].str.strip().str.lower()

    # Standardize sex values
    if 'sex' in df.columns:
        df['sex'] = df['sex'].replace({
            'm': 'male',
            'f': 'female'
        })

    # Convert numerical columns
    numeric_columns = [
        'age', 'fare', 'sibsp', 'parch', 'pclass', 'survived'
    ]

    for column in numeric_columns:
        if column in df.columns:
            df[column] = pd.to_numeric(
                df[column], errors='coerce'
            )


    # Handle impossible values
    for column in ['age', 'fare', 'sibsp', 'parch']:
        if column in df.columns:
            df.loc[df[column] < 0, column] = np.nan

    # Create age missing indicator
    if 'age' in df.columns:
        df['age_missing'] = df['age'].isna()

    # Impute numerical columns with median
    for column in df.select_dtypes(include='number').columns:
        if df[column].isna().any():
            df[column] = df[column].fillna(df[column].median())

    # Impute categorical columns with mode
    for column in df.select_dtypes(include='object').columns:
        if df[column].isna().any():
            df[column] = df[column].fillna(df[column].mode()[0])

    # Convert low-cardinality columns to category
    for column in df.select_dtypes(include='object').columns:
        if df[column].nunique() <= 10:
            df[column] = df[column].astype('category')

    return df

CLEAN FUNCTION


In [30]:
print_title('APPLY CLEAN FUNCTION')

df_clean = clean(df)

print(f'Original shape: {df.shape}')
print(f'Cleaned shape: {df_clean.shape}')

APPLY CLEAN FUNCTION
Original shape: (891, 15)
Cleaned shape: (891, 16)
